# Brain Stroke CT Baseline

This notebook clones the baseline branch, runs the 15-epoch U-Net training, evaluates the held-out split, and writes external-test masks to `/kaggle/working`.

Progress appears live in the training cell. The same progress is persisted in `reports/experiments/baseline/run.log` and `reports/experiments/baseline/progress.json`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/kaggle/working/brain-stroke-ct-segmentation')
repo_url = 'https://github.com/echo14670/brain-stroke-ct-segmentation.git'
branch = 'baseline/core'
if not repo.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(repo)],
        check=True,
    )

candidates = [
    path for path in Path('/kaggle/input').rglob('Brain_Stroke_CT_Dataset')
    if all((path / name).is_dir() for name in ('Bleeding', 'Ischemia', 'Normal', 'External_Test'))
]
if not candidates:
    raise FileNotFoundError('Brain_Stroke_CT_Dataset was not mounted')
data_root = candidates[0]
os.environ['BSC_DATA_ROOT'] = str(data_root)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
print('repository:', repo)
print('dataset:', data_root)
print('python:', sys.version)

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/train.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/evaluate.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)
subprocess.run(
    [sys.executable, 'scripts/infer.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)

In [ ]:
results = repo / 'reports' / 'experiments' / 'baseline'
print((results / 'summary.md').read_text(encoding='utf-8'))
(Path('/kaggle/working') / 'BASELINE_RUN_COMPLETE').write_text('complete\n', encoding='utf-8')
print('result directory:', results)